# Model Fairness: Good Overall Performance Can Hide Unequal Errors

## One-hour machine learning case study

A company uses a model to recommend candidates for interviews. Overall accuracy appears acceptable, but qualified candidates from one group are missed more often than qualified candidates from another.

This notebook examines subgroup evaluation, false-negative rates, equal opportunity, proxy variables, and the limits of purely technical fairness fixes.

> **Central question:** Who experiences the model's errors?

## Learning objectives

Students will:

- calculate confusion-matrix metrics by subgroup;
- identify when aggregate performance hides unequal outcomes;
- distinguish performance disparity from a complete explanation of bias;
- examine proxy variables and historical labels;
- compare fairness goals that may conflict;
- recommend technical and organizational safeguards;
- use AI to surface missing stakeholder perspectives;
- communicate findings without overclaiming.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix

rng = np.random.default_rng(21)

def create_group(group, n, qualified_rate, tpr, fpr):
    qualified = rng.binomial(1, qualified_rate, n)
    recommended = np.zeros(n, dtype=int)

    positive_idx = qualified == 1
    negative_idx = qualified == 0

    recommended[positive_idx] = rng.binomial(1, tpr, positive_idx.sum())
    recommended[negative_idx] = rng.binomial(1, fpr, negative_idx.sum())

    return pd.DataFrame({
        "group": group,
        "qualified": qualified,
        "recommended": recommended
    })

group_a = create_group("Group A", 1200, qualified_rate=0.50, tpr=0.82, fpr=0.18)
group_b = create_group("Group B", 800, qualified_rate=0.50, tpr=0.58, fpr=0.12)

df = pd.concat([group_a, group_b], ignore_index=True)
df.head()

# Part 1 — Overall performance

First, evaluate the model without examining subgroups.

In [ ]:
def subgroup_metrics(data):
    tn, fp, fn, tp = confusion_matrix(
        data["qualified"], data["recommended"], labels=[0, 1]
    ).ravel()

    return pd.Series({
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "Accuracy": (tp + tn) / (tp + fp + tn + fn),
        "Recall / TPR": tp / (tp + fn),
        "False-negative rate": fn / (tp + fn),
        "False-positive rate": fp / (fp + tn),
        "Precision": tp / (tp + fp)
    })

overall = subgroup_metrics(df)
overall.to_frame("Overall").style.format({
    "Accuracy": "{:.1%}",
    "Recall / TPR": "{:.1%}",
    "False-negative rate": "{:.1%}",
    "False-positive rate": "{:.1%}",
    "Precision": "{:.1%}"
})

## Initial judgment

Based only on overall performance:

1. Would you approve the model?
2. Which errors matter most in hiring?
3. What subgroup information should be requested?
4. Who may be harmed by a false negative?

# Part 2 — Evaluate each group

In [ ]:
group_results = df.groupby("group").apply(subgroup_metrics, include_groups=False)
group_results

In [ ]:
rate_columns = [
    "Accuracy", "Recall / TPR", "False-negative rate",
    "False-positive rate", "Precision"
]

group_results[rate_columns].style.format("{:.1%}")

In [ ]:
plot_rates = group_results[["Recall / TPR", "False-negative rate"]]

plot_rates.plot(kind="bar", figsize=(8, 5))
plt.ylim(0, 1)
plt.ylabel("Rate")
plt.xlabel("Candidate group")
plt.title("Qualified-candidate outcomes by group")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()

## Interpretation

1. Which group has the higher false-negative rate?
2. What does that mean in plain language?
3. Can the overall accuracy reveal this problem?
4. Does the disparity prove intentional discrimination?
5. What investigations should follow?

# Part 3 — Examine the label

The target variable is “qualified,” but how was qualification defined?

Possible labels include:

- historical hiring decisions;
- manager ratings;
- performance after hire;
- a skills assessment;
- whether the candidate stayed for one year.

Each label embeds different assumptions and possible historical biases.

Discuss:

1. Who created the label?
2. Could past decisions reflect unequal opportunity?
3. Does the label measure job ability or access to opportunity?
4. What information may be missing?

# Part 4 — Proxy variables

A protected characteristic may be excluded while related variables remain.

Possible proxies include:

- ZIP code;
- school attended;
- employment gaps;
- name;
- commute distance;
- participation in certain organizations.

Removing one column does not automatically remove the underlying signal.

Explain why “fairness through unawareness” may be insufficient.

# Part 5 — Fairness goals can conflict

Different fairness goals include:

- similar true-positive rates;
- similar false-positive rates;
- similar precision;
- similar selection rates;
- calibrated probabilities.

These conditions cannot always be satisfied simultaneously, especially when underlying outcome rates differ.

The appropriate goal depends on the use case, law, ethics, and consequences—not only mathematics.

## AI as a stakeholder challenger

Use one prompt:

> Act as a rejected qualified candidate. What questions would you ask about this model?

> Act as an HR leader. What operational evidence is needed before deployment?

> Identify possible proxy variables in a hiring model.

> Challenge the claim that removing protected attributes makes a model fair.

Evaluate:

**Perspective AI added:**  

**Claim requiring verification:**  

**Stakeholder still missing:**  

**Action the organization should take:**

# Part 6 — Build a responsible recommendation

Your recommendation must include:

### Technical actions
- subgroup metrics;
- threshold review;
- label audit;
- missing-data analysis;
- ongoing monitoring.

### Process actions
- structured human review;
- appeal process;
- documentation;
- periodic external audit;
- candidate communication.

### Limits
State what the current analysis cannot prove.

### Recommendation

**Technical findings:**  

**Immediate safeguard:**  

**Additional data needed:**  

**Human oversight:**  

**What cannot yet be concluded:**

# Transfer task — Student early-alert model

A college model has equal overall accuracy for two student groups, but recall is 88% for one group and 59% for another.

Answer:

1. What does the recall gap mean?
2. Why might equal accuracy hide the problem?
3. What should be reviewed in the training labels?
4. What intervention safeguards are needed?
5. Why should the model not make automatic academic decisions?

# Exit reflection

- Overall performance can hide …
- A false negative in this case means …
- Removing protected attributes is insufficient because …
- A fairness metric is a choice about …
- Technical analysis must be combined with …

# Instructor checklist

- [ ] Students computed subgroup metrics.
- [ ] Students translated disparities into human consequences.
- [ ] Students avoided claiming that disparity alone proves cause.
- [ ] Students audited the target label.
- [ ] Students examined proxy variables.
- [ ] Students recognized conflicting fairness goals.
- [ ] AI introduced stakeholder perspectives.
- [ ] Recommendations included technical and organizational controls.

# Closing principle

A model is not adequately evaluated until we know how its errors are distributed.

The central question is not only, “How often is the model correct?” It is also, “For whom does it fail, and what happens when it does?”